In [ ]:
import timesfm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm 
import os 
from itertools import product

from timesfm_functions import (
    TimesFMModel,
    load_aluminium_data,
    calculate_prediction_metrics
)

from functions import line_plot, mape, mae, rmse, mse, mase, pred_value_to_char

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


/home/panstenos/.pyenv/versions/3.11.10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded PyTorch TimesFM, likely because python version is 3.11.10 (main, Aug 20 2025, 22:27:03) [GCC 13.3.0].


In [2]:
def calculate_prediction_metrics(predictions, true_values):
    """
    Calculate common error metrics to evaluate prediction accuracy.

    Parameters
    ----------
    predictions : array-like
        The predicted values from a model.
    true_values : array-like
        The actual observed/true values.

    Returns
    -------
    dict
        A dictionary containing:
        - "MAPE": Mean Absolute Percentage Error
            Measures prediction accuracy as a percentage; lower is better.
        - "MAE": Mean Absolute Error
            Average of absolute differences between predictions and true values.
        - "RMSE": Root Mean Squared Error
            Square root of average squared differences; penalizes large errors.
        - "MSE": Mean Squared Error
            Average of squared differences; sensitive to outliers.
        - "MASE": Mean Absolute Scaled Error
            Scale-independent error metric useful for comparing across datasets.
    """

    # Coerce to numpy arrays
    y_pred = np.asarray(predictions)
    y_true = np.asarray(true_values)

    # Make them column vectors if 1D
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)
    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)

    # Align lengths (guard against off-by-one slicing)
    n = min(len(y_pred), len(y_true))
    if n == 0:
        raise ValueError("Empty inputs: predictions/true_values have no overlapping length.")
    y_pred = y_pred[:n]
    y_true = y_true[:n]


    mape_val = mape(true_values, predictions)
    mae_val = mae(true_values, predictions)
    rmse_val = rmse(true_values, predictions)
    mse_val = mse(true_values, predictions)
    mase_val = mase(true_values, predictions) 

    return {
        "MAPE": mape_val,
        "MAE": mae_val,
        "RMSE": rmse_val,
        "MSE": mse_val,
        "MASE": mase_val
    }

In [3]:
df = load_aluminium_data()
df.head()

,al_lme_prices_log_returns,al_lme_prices_daily_returns,us_dollar_index,canadian_dollar_spot,emirate_dirham_spot,russian_ruble_spot,nor_krone_spot,australian_dollar_spot,malaysian_ringgit_spot,euro_spot,...,us_gdp_agr,china_caixin_pmi,germany_pmi,japan_pmi,us_pmi,1w_vol,1m_vol,3m_vol,1y_vol,al_lme_prices_abs_log_returns
date,,,,,,,,,,,,,,,,,,,,,
1/18/2016,0.009106,0.009148,98.956,1.4561,3.6726,79.3775,8.8984,1.457089,4.3930,0.918274,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.009106
1/19/2016,-0.004711,-0.004700,98.991,1.4576,3.6723,78.7850,8.8061,1.448646,4.3625,0.917011,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.004711
1/20/2016,-0.012730,-0.012650,99.091,1.4501,3.6723,81.3475,8.8728,1.448016,4.3910,0.918358,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.012730
1/21/2016,0.015257,0.015374,99.056,1.4261,3.6722,82.3675,8.7823,1.429388,4.3730,0.919794,...,2.0,48.2,53.2,52.6,51.2,NaN,NaN,NaN,NaN,0.015257
1/22/2016,-0.002189,-0.002187,99.574,1.4115,3.6722,78.1000,8.7341,1.428571,4.2940,0.926441,...,2.0,48.2,53.2,52.6,51.2,0.177532,NaN,NaN,NaN,0.002189


In [7]:
# make dirs to save results
base_dir = "timesfm_vanilla_results"
plots_dir = os.path.join(base_dir, "plots")
metrics_dir = os.path.join(base_dir, "metrics")
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

# main_run
def run(expiry, window_size, normalize, freq_input):
    X = df[f'{pred_value_to_char(expiry)}_vol'].dropna()
    model = TimesFMModel(expiry=expiry, context_length=window_size)


    preds = []
    for i in tqdm(range(len(X) - window_size - expiry + 1)):
        mean_pred, quantiles = model.predict(
            forecast_input=X[i:i+window_size],
            frequency_input=freq_input,
            normalize=normalize
        )
        preds.append(mean_pred[-1])

    preds = np.asarray(preds)
    true = X[window_size + expiry - 1 : window_size + expiry - 1 + len(preds)].to_numpy()

    # plot + save (reuse your existing line_plot + save path logic)
    ax, fig = line_plot(range(len(preds[:300])), preds[:300], 'pred', show=False)
    _, _   = line_plot(range(len(true[:300])),  true[:300],  'true', linecolor='red', ax=ax, show=False)
    fig.savefig(os.path.join(plots_dir, f"pred_vs_true_{pred_value_to_char(expiry)}_window{window_size}_freq{freq_input[0]}_normal{normalize}.png"), dpi=150, bbox_inches="tight")

    # metrics
    metrics = calculate_prediction_metrics(preds, true)


    return [
        expiry, 
        window_size, 
        int(normalize), 
        freq_input[0],
        metrics["MAPE"], 
        metrics["MAE"], 
        metrics["RMSE"], 
        metrics["MSE"], 
        metrics["MASE"]
        ]

In [ ]:
expiries      = [5, 22, 66, 252]
window_sizes  = [32, 64, 128, 256, 512, 1024]
normalizes    = [True, False]
freq_inputs   = [[0], [1], [2]]

rows = []
for expiry, window_size, normalize, freq_input in product(expiries, window_sizes, normalizes, freq_inputs):
    if expiry // 2 > window_size:
        continue
    print(f"expiry={expiry}, window_size={window_size}, normalize={normalize}, freq_input={freq_input}")
    row = run(expiry, window_size, normalize, freq_input)
    rows.append(row)

metrics_csv = os.path.join(metrics_dir, "metrics.csv")
metrics_df = pd.DataFrame(rows, columns=["expiry","window_size","normalize","freq_input","mape","mae","rmse","mse","mase"])
metrics_df.to_csv(metrics_csv, index=False)